# 🚀 Process Monitor Dashboard

### **IMPORTANT: PLEASE RELOAD THIS PAGE (F5) AND RUN THE CELL BELOW**

This dashboard is now consolidated into a single cell for the best experience. 

1. **Refresh/Reload** your browser page to ensure you have the latest version.
2. Click the cell below and press **Shift + Enter** (or click the Run button).
3. Click the **Start Dashboard** button that appears.

In [1]:
# === CONSOLIDATED PROCESS MONITOR DASHBOARD ===
import os
import sys
import time
import subprocess

def setup_and_run():
    print("📦 Checking environment...")
    
    # 1. Auto-install missing dependencies
    try:
        import pandas as pd
        import ipywidgets as widgets
        from IPython.display import display, clear_output
    except ImportError:
        print("⚠️ Missing libraries. Installing pandas and ipywidgets...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "ipywidgets"])
        print("✅ Installed. PLEASE RESTART KERNEL AND RUN AGAIN.")
        return

    # 2. Path Setup
    project_root = os.path.abspath(".")
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

    try:
        from gui.notebook_interface import fetch_processes, processes_to_dataframe, filter_processes, sort_processes, kill_process, pause_process, resume_process, set_priority, _style_dataframe
        print("🔌 Backend bridge connected.")
    except Exception as e:
        print(f"❌ Fatal Error: Could not load backend bridge: {e}")
        return

    # --- UI Components ---
    refresh_interval = 3
    is_running = {'val': False} # Use dict for mutability in nested functions

    dropdown = widgets.Dropdown(description="Process:", layout={'width': '500px'})
    kill_btn = widgets.Button(description="Kill", button_style='danger', icon='trash')
    pause_btn = widgets.Button(description="Pause", icon='pause')
    resume_btn = widgets.Button(description="Resume", icon='play')
    priority_slider = widgets.IntSlider(min=-10, max=10, value=0, description="Priority")
    set_priority_btn = widgets.Button(description="Set Priority", button_style='info')
    auto_toggle = widgets.ToggleButton(value=False, description="Auto Control", button_style='warning', icon='robot')
    status_label = widgets.Label(value="Status: Ready")
    start_btn = widgets.Button(description="Start Dashboard", button_style='success', icon='play-circle')
    stop_btn = widgets.Button(description="Stop Dashboard", button_style='danger', icon='stop-circle')
    output_area = widgets.Output()

    # --- Layout ---
    header = widgets.HTML("""
        <div style='background-color: #2e86de; padding: 15px; border-radius: 10px; margin-bottom: 20px;'>
            <h1 style='color: white; margin: 0;'>🚀 Process Monitor</h1>
            <p style='color: #dff9fb; margin: 5px 0 0 0;'>Real-time system process management</p>
        </div>
    """)
    
    controls = widgets.VBox([
        widgets.HBox([start_btn, stop_btn]),
        dropdown,
        widgets.HBox([kill_btn, pause_btn, resume_btn]),
        widgets.HBox([priority_slider, set_priority_btn]),
        auto_toggle,
        status_label
    ], layout={'padding': '15px', 'border': '1px solid #ddd', 'border-radius': '8px', 'margin-bottom': '20px'})

    dashboard_ui = widgets.VBox([header, controls, output_area])

    # --- Logic ---
    def update_display():
        try:
            processes = fetch_processes()
            df = processes_to_dataframe(processes)
            df = filter_processes(df)
            df = sort_processes(df)
            
            if not df.empty:
                dropdown.options = [
                    (f"{row['Name']} (PID {row['PID']}) - CPU {row['CPU']}%", row['PID'])
                    for _, row in df.iterrows()
                ]
            
            with output_area:
                clear_output(wait=True)
                try:
                    display(_style_dataframe(df))
                except:
                    display(df)
            
            if auto_toggle.value:
                for _, row in df.iterrows():
                    if row["CPU"] > 40.0:
                        pause_process(row["PID"])
        except Exception as e:
            status_label.value = f"Status: Update Error - {e}"

    def on_action(b):
        if not dropdown.value: return
        try:
            if b.description == "Kill": kill_process(dropdown.value)
            elif b.description == "Pause": pause_process(dropdown.value)
            elif b.description == "Resume": resume_process(dropdown.value)
            elif b.description == "Set Priority": set_priority(dropdown.value, priority_slider.value)
            status_label.value = f"Status: Executed {b.description} on PID {dropdown.value}"
        except Exception as e:
            status_label.value = f"Status: Action Error - {e}"

    def start_loop(b):
        if is_running['val']: return
        is_running['val'] = True
        start_btn.disabled = True
        status_label.value = "Status: Running..."
        while is_running['val']:
            update_display()
            time.sleep(refresh_interval)
            # Manual check for stop since Jupyter loop can be tricky
            if not is_running['val']: break

    def stop_loop(b):
        is_running['val'] = False
        start_btn.disabled = False
        status_label.value = "Status: Stopped"

    # Bindings
    for btn in [kill_btn, pause_btn, resume_btn, set_priority_btn]:
        btn.on_click(on_action)
    start_btn.on_click(start_loop)
    stop_btn.on_click(stop_loop)

    # Initial display
    display(dashboard_ui)
    print("✅ Dashboard Loaded. Click 'Start Dashboard' to begin.")

setup_and_run()

📦 Checking environment...
🔌 Backend bridge connected.


✅ Dashboard Loaded. Click 'Start Dashboard' to begin.
